# Bloc A – Les annulations

## Le fil rouge
Une seule question derrière tout le bloc : **qu'est-ce qui fait qu'un client annule, et peut-on le prévoir ?**
- **Q1** cherche *quelles* informations comptent.
- **Q2** étudie la plus importante : le délai.
- **Q5** s'en sert pour *prévoir* les annulations.

**Repère pour tout le bloc** : 26 % des réservations sont annulées. Chaque taux se compare à ce chiffre.

---

## Q1 – Qu'est-ce qui distingue une annulation ?

**Point de départ** : deux avis s'opposent.
- Le revenue manager : « ce sont les plateformes (OTA) ».
- Les directeurs : « c'est la clientèle loisir ».

**Logique** : on ne croit personne, on vérifie avec les données.

**Étape 1 – Tout comparer**
- On compare chaque colonne entre réservations annulées et honorées.
- Résultat : **seul le délai change vraiment** (52 jours contre 22).
- → On sait déjà où chercher.

**Preuve 1 – Les taux par canal et par segment**
- OTA : 27,7 %, à peine au-dessus de la moyenne.
- Affaires 16 % contre loisir et groupe 33 % : un vrai écart.
- → Le segment compte plus que le canal.

**Preuve 2 – À segment égal**
- Pourquoi : si les OTA avaient surtout des clients loisir, elles paraîtraient « mauvaises » à cause de leurs clients.
- On compare donc les canaux entre clients du même type.
- Résultat : chez les affaires, les OTA annulent même *moins*.
- → Le canal n'a pas d'effet régulier.

**Preuve 3 – À délai égal**
- Pourquoi : les clients loisir réservent beaucoup plus tôt (55 jours contre 9).
- On compare les canaux entre réservations faites aussi longtemps à l'avance.
- Résultat : les OTA sont dans la moyenne… sauf au-delà de 60 jours (48 % contre 34 %).
- → Le vrai risque : réserver très tôt, surtout sur une plateforme.

**Conclusion** : ce n'est pas le canal, c'est surtout **le délai de réservation**.

---

## Q2 – Réserver tôt, est-ce sûr ?

**Logique** : la Q1 dit *que* le délai compte. La Q2 dit *comment* et *à partir de quand*.

- On découpe le délai en 5 tranches (de 0-7 jours à plus de 120 jours).
- Résultat : le taux monte à chaque tranche, **de 16 % à 67 %**.
- Vérification à segment égal : l'effet existe aussi chez les clients loisir et groupe.
- → **C'est l'inverse de l'idée reçue.**
- → Le risque devient fort **au-delà de 60 jours** : c'est là qu'il faut agir.

---


## Q5 – Peut-on prédire une annulation ? (kNN)
1. **Apprentissage / test** → on juge le modèle sur des données qu'il n'a pas vues.
2. **Choisir k avec l'AUC** (validation croisée) → k = 51.
3. **Déplacer le seuil** → à 0,50, le modèle ne trouve presque rien ; vers 0,20-0,30, il trouve plus de la moitié des annulations.
4. **Règle à la main** (délai > 45 j) → fait aussi bien : le délai porte toute l'information.
5. **Export** → une probabilité par réservation pour le bloc C.


---

## Ce qu'on recommande
- Ne pas pénaliser les plateformes.
- Acompte ou tarif non remboursable **au-delà de 60 jours**.
- Utiliser le modèle pour estimer le nombre d'annulations **par hôtel**, pas pour juger un client.

In [ ]:
#les imports
import glob                       # trouver les fichiers dans Colab
import pandas as pd               # tableaux : groupby, pivot_table, merge, cut
import matplotlib.pyplot as plt   # graphiques

#les imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

#les imports pour choisir le k de knn
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

In [ ]:
df = pd.read_csv(glob.glob('/content/**/reservations_modelisation_794*.csv', recursive=True)[0])
df['annule'] = df['a_annule_bin']
modele = df   # même fichier pour la Q5
print(len(df), 'réservations | taux :', round(df['annule'].mean() * 100, 1), '%')
display((df.groupby('categorie')['annule'].mean() * 100).round(1))

Catégorie d’hôtel : le taux d’annulation va de 24,9 % (3 étoiles) à 27,5 % (4 étoiles), proche de la moyenne de 26,1 %. La catégorie ne distingue pas les annulations.

**ÉTAPE 1 : comparer selon la cible**

une seule colonne change vraiment, le délai.

Les réservations annulées ont été faites en médiane 52 jours avant l’arrivée, contre 22 pour les honorées.
Le nombre de personnes change un peu (lié au segment : les groupes sont plus nombreux). Le reste ne bouge pas


In [ ]:
# ÉTAPE 1 : comparer selon la cible (séance 2)
colonnes = ['delai_reservation_j', 'nb_nuits', 'nb_personnes',
            'prix_nuit_eur', 'mois_sejour', 'nuitees_cumulees_client']

comparaison = df.groupby('annule')[colonnes].median().T
comparaison.columns = ['honorées', 'annulées']
comparaison['écart'] = comparaison['annulées'] - comparaison['honorées']
display(comparaison.round(1))

# Les colonnes texte : taux d'annulation par modalité (%)
for col in ['weekend', 'petit_dejeuner']:
    print(col)
    display((df.groupby(col)['annule'].mean() * 100).round(1))

Ce que demande la direction
* **Q1** : qu'est-ce qui distingue une réservation annulée d'une réservation honorée ?
* **Q2** : un client qui réserve tôt est-il un client sûr ?


* **Q5** : au moment de la réservation, peut-on prédire si elle sera annulée ?





**Données :** reservations_modelisation_794.csv (794 réservations, anomalies retirées), pour toutes les questions.

**Cible :** a_annule (oui / non). On la transforme en 1 / 0 pour calculer des taux. Taux global d'annulation : 26 %. C'est le point de comparaison de tous les taux qui suiven

### Q1 – Compréhension
Deux explications s'opposent dans l'entreprise :
- **H1 (revenue manager)** : les annulations viennent des plateformes (canal OTA).
- **H2 (directeurs d'hôtel)** : les annulations viennent de la clientèle loisir.
Comme le recommande le cours (séance 2), on écrit les hypothèses **avant** l'analyse, pour ne pas trouver seulement ce qu'on cherchait.

**Méthode** : comparer le taux d'annulation selon chaque groupe, puis lire l'écart avec le taux global (séance 2, « comparer selon la cible »). On regarde toujours l'effectif à côté du taux : un taux calculé sur peu de réservations est fragile.


In [ ]:
print('Le piège : répartition des canaux dans chaque segment')
display(pd.crosstab(df['segment_declare'], df['canal'], normalize='index').round(2))

print('La variable cachée : délai médian de réservation par segment (jours)')
display(df.groupby('segment_declare')['delai_reservation_j'].median())


| Canal | Taux | Réservations |
|---|---|---|
| OTA | 28 % | 339 |
| Direct | 27 % | 212 |
| Entreprise | 24 % | 157 |
| Agence | 22 % | 92 |

Les OTA ne sont que 2 points au-dessus de la moyenne (26 %), et le canal direct fait presque pareil (27 %). L'écart maximal entre canaux est de 6 points, et l'agence ne compte que 92 réservations. **H1 n'est pas confirmée à ce stade.**


**comparer les canaux entre clients du même type:)**


In [ ]:
display((df.pivot_table(index='segment_declare', columns='canal',
                        values='annule', aggfunc='mean') * 100).round(0))

L’hypothèse du revenue manager n’est pas confirmée. Les OTA ont un taux d’annulation de 28 %, proche de la moyenne (26 %) et du canal direct (27 %). À segment égal, l’effet des plateformes n’est pas régulier : il disparaît chez les clients affaires. Ce qui distingue les annulations, c’est le segment, et derrière lui le délai de réservation.

**Preuve 01: lecart est faible entre les annulations OTA et le reste**

In [ ]:
#calcule l'ecart par canal
par_canal = df.groupby('canal')['annule']
taux = par_canal.mean() * 100
size = par_canal.size()
nb_annul = par_canal.sum()
taux_global = df['annule'].mean() * 100
ecart_global = taux - taux_global

results_canal = pd.DataFrame(
    {
        'taux': taux.round(1),
        'effectifs': size,
        'annulations': nb_annul,
        'ecart_global': ecart_global.round(1)

    }
)



#calcule size et mean par segment
par_segment_declare = df.groupby('segment_declare')['annule']
taux_seg = par_segment_declare.mean() * 100
size_seg = par_segment_declare.size()
nb_annul_seg = par_segment_declare.sum()
ecart_global_seg = taux_seg - taux_global


results_segment_declare = pd.DataFrame(
    {
        'taux': taux_seg.round(1),
        'effectifs': size_seg,
        'annulations': nb_annul_seg,
        'ecart_global_seg': ecart_global_seg.round(1)

    }
)



#afficher les resultats

display(results_canal)
display(results_segment_declare)




LECTURE – Preuve 1 : comparer les taux d'annulation

Taux global d'annulation : 26,1 % (209 annulations sur 800).
C'est la référence.

Par canal (hypothèse du revenue manager) :
- OTA : 27,7 %, soit seulement +1,6 point au-dessus de la moyenne.
- Le canal direct est presque au même niveau (26,9 %).
- Écart entre le canal le plus haut (OTA) et le plus bas
   (agence, 21,7 %) : seulement 6 points, et l'agence ne compte
   que 92 réservations.

 Par segment (hypothèse des directeurs) :
 - Affaires : 16,2 % (-9,9 points), ils annulent beaucoup moins.
 - Loisir (32,5 %) et groupe (33,3 %) annulent deux fois plus.
 - Écart entre segments : 17 points, près de 3 fois celui
   entre canaux.

 Conclusion provisoire :
 Les plateformes annulent à peine plus que la moyenne :
 l'hypothèse du revenue manager n'est PAS confirmée à ce stade.
 Le segment fait une différence beaucoup plus nette, ce qui va
 dans le sens des directeurs.
 À vérifier : canal et segment ne se mélangent-ils pas ?
 -> preuve 2 (à segment égal).


**PREUVE 2 : comparer les canaux à segment égal**

In [ ]:
# PREUVE 2 : comparer les canaux à segment égal

# Taux d'annulation (%) pour chaque segment × canal
taux_croise = (df.pivot_table(index='segment_declare', columns='canal',
                              values='annule', aggfunc='mean') * 100).round(0)

# Nombre de réservations dans chaque case
effectifs_croise = df.pivot_table(
    index='segment_declare',
    columns='canal',
    values='annule',
    aggfunc='size'
                                  )

print("Taux d'annulation segment × canal (%) :")
display(taux_croise)
print('Effectifs :')
display(effectifs_croise)

**Lecture – Preuve 2 (à segment égal)**

Si les plateformes étaient le problème, les OTA annuleraient plus que les autres canaux dans chaque type de client.

**Ce n’est pas le cas**: chez les clients affaires, les OTA annulent même moins (14 %) que le direct (20 %). Chez les loisirs et les groupes, elles sont à peine au-dessus.

En revanche, dans tous les canaux, les clients affaires annulent beaucoup moins que les clients loisir et groupe.

**Conclusion** : ce n’est pas le canal qui explique les annulations, c’est le type de client.

In [ ]:
# PREUVE 3 : comparer les canaux à délai égal
df['tranche'] = pd.cut(df['delai_reservation_j'], bins=[-1, 30, 60, 999],
                       labels=['0-30 j', '31-60 j', '> 60 j'])

print("Taux d'annulation tranche × canal (%) :")
display((df.pivot_table(index='tranche', columns='canal', values='annule',
                        aggfunc='mean', observed=True) * 100).round(0))
print('Effectifs :')
display(df.pivot_table(index='tranche', columns='canal', values='annule',
                       aggfunc='size', observed=True))

# Écart des OTA par rapport aux autres canaux, dans chaque tranche
ota = df[df['canal'] == 'OTA'].groupby('tranche', observed=True)['annule'].mean() * 100
autres = df[df['canal'] != 'OTA'].groupby('tranche', observed=True)['annule'].mean() * 100
ecarts = pd.DataFrame({'OTA': ota, 'autres canaux': autres, 'écart': ota - autres})
display(ecarts.round(1))



#lautre sense de le delai et le canal

tableau_canal_delai = (df.pivot_table(index='canal', columns='tranche', values='annule',
                                      aggfunc='mean', observed=True) * 100).round(0)

tableau_canal_delai['écart'] = tableau_canal_delai['> 60 j'] - tableau_canal_delai['0-30 j']

display(tableau_canal_delai[['0-30 j', '> 60 j']])

**remarque**: Les plateformes ne sont pas le problème en général : le risque vient des réservations faites très tôt, et il est un peu plus fort quand elles passent par une plateforme, probablement parce que l'annulation y est facile et gratuite.

In [ ]:
#decoupe le delai en tranches
df['tranche_delai'] = pd.cut(df['delai_reservation_j'],
                             bins=[-1, 7, 30, 60, 120, 999],
                             labels=['0-7 j', '8-30 j', '31-60 j', '61-120 j', '> 120 j'])

#calcule taux, effectif, annulations et ecart par tranche
par_tranche = df.groupby('tranche_delai', observed=True)['annule']
taux_tranche = par_tranche.mean() * 100
size_tranche = par_tranche.size()
nb_annul_tranche = par_tranche.sum()
ecart_tranche = taux_tranche - taux_global

results_tranche = pd.DataFrame(
    {
        'taux': taux_tranche.round(1),
        'effectifs': size_tranche,
        'annulations': nb_annul_tranche,
        'ecart_global': ecart_tranche.round(1)
    }
)

display(results_tranche)

Lecture : le taux d’annulation monte à chaque tranche, de 16 % (réservé moins d’une semaine avant) à 67 % (plus de 4 mois avant). Au-delà de 60 jours, il dépasse nettement la moyenne (26 %). La dernière tranche ne compte que 46 réservations, mais la hausse est régulière sur toutes les tranches.

In [ ]:
#taux d'annulation par tranche et par segment
taux_tranche_seg = (df.pivot_table(index='tranche_delai', columns='segment_declare',
                                   values='annule', aggfunc='mean', observed=True) * 100).round(0)
size_tranche_seg = df.pivot_table(index='tranche_delai', columns='segment_declare',
                                  values='annule', aggfunc='size', observed=True)

display(taux_tranche_seg)
display(size_tranche_seg)

Vérification : les clients affaires réservent presque tous moins d’un mois avant, donc on ne peut pas tester l’effet du délai chez eux. Chez les clients loisir et groupe, le taux monte aussi avec le délai (environ 30 % avant 2 mois, plus de 60 % au-delà de 4 mois). L’effet du délai existe donc même à segment égal.

In [ ]:
#graphique du taux par tranche
plt.figure(figsize=(7, 4))
plt.bar(results_tranche.index.astype(str), results_tranche['taux'], color='#C44E52')
plt.axhline(taux_global, color='grey', linestyle='--', label='moyenne')
plt.ylabel("% de réservations annulées")
plt.xlabel("Délai entre réservation et arrivée")
plt.title("Plus on réserve tôt, plus on annule")
plt.legend()
plt.show()

Conclusion Q2 : c’est l’inverse de l’idée reçue. Plus un client réserve tôt, plus il annule : 16 % à moins d’une semaine, 67 % au-delà de 4 mois. L’effet se retrouve même à segment égal.

Recommandation Q2 : pour les réservations faites plus de 60 jours avant l’arrivée, demander un acompte ou proposer un tarif non remboursable (avec une petite remise), ou fixer une date limite d’annulation gratuite (par exemple 30 jours avant l’arrivée).

**Q5:le modele de prédiction it's with knn:**

**Pourquoi KNN:**

*   prédire oui ou non (annulé ou pas) : c'est une classification dune categorie
*   Il donne une probabilité(important pour le block C)






**TODO:**

pour choisir le k
*   le F1 score
*   le seuil












In [ ]:
#Q5 : on utilise le fichier sans les anomalies (365 nuits, prix à 0 €)
chemin_modele = glob.glob('/content/**/reservations_modelisation_794*.csv', recursive=True)[0]
modele = pd.read_csv(chemin_modele)
modele['annule'] = (modele['a_annule'] == 'oui').astype(int)

print('Réservations :', len(modele))
print('Annulations :', modele['annule'].sum())
print('Taux :', round(modele['annule'].mean() * 100, 1), '%')

In [ ]:
#X = les informations connues au moment de la réservation (colonnes numériques)
#y = ce qu'on veut prédire
X = modele[['delai_reservation_j', 'nb_nuits', 'nb_personnes', 'prix_nuit_eur']]
y = modele['annule']

#70 % pour apprendre, 30 % pour tester, comme dans le cours (séance 3)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

print('Apprentissage :', X_train.shape, '| Test :', X_test.shape)
print('Taux d\'annulation apprentissage / test :',
      round(y_train.mean() * 100, 1), '/', round(y_test.mean() * 100, 1))

In [ ]:
#on teste plusieurs k par validation croisée, sur l'apprentissage seulement

cv = StratifiedKFold(5, shuffle=True, random_state=42)

for k in [5, 15, 31, 51, 71, 101]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(k))
    scores = cross_val_score(knn, X_train[['delai_reservation_j']], y_train,
                             cv=cv, scoring='roc_auc')
    print(f'k = {k:3d} | AUC = {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(51))
knn.fit(X_train[['delai_reservation_j']], y_train)
proba = knn.predict_proba(X_test[['delai_reservation_j']])[:, 1]
print('AUC sur le test :', round(roc_auc_score(y_test, proba), 3))

for s in [0.50, 0.40, 0.35, 0.30, 0.25, 0.20]:
    alerte = (proba >= s).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_test, alerte).ravel()
    print(f'seuil {s:.2f} | signalées {alerte.sum():3d} | trouvées {vp:2d} | '
          f'rappel {recall_score(y_test, alerte):.2f} | '
          f'précision {precision_score(y_test, alerte, zero_division=0):.2f} | '
          f'F1 {f1_score(y_test, alerte):.2f}')

**visualization des prédictions:**
vous pouvez chnager le (nouveau_delai) pour predire le taux d'annulation.
le k=51 c le resultats de la validation crosée


Choix de k : on garde les vraies proportions (26 % d'annulations) et on choisit k avec l'AUC en validation croisée, car elle ne dépend d'aucun seuil (cours « Évaluer un modèle »). L'AUC monte de 0,57 (k = 5) à environ 0,63 (k = 71 à 101). Au-delà de 51, les écarts (0,01 à 0,02) sont plus petits que l'écart-type (≈ 0,06) : ils ne sont pas fiables. On retient k = 51, le plus petit k du plateau.

Le seuil : à 0,50 (le seuil par défaut de predict), le modèle ne signale que 5 réservations et trouve 3 annulations sur 62 : le rappel est de 0,05. En baissant le seuil, on trouve beaucoup plus d'annulations (33 à 0,30, 50 à 0,20), et le F1 passe de 0,09 à environ 0,45. Le seuil ne se choisit pas avec une métrique mais avec les coûts : rater une annulation laisse une chambre vide, alors qu'une fausse alerte coûte peu (par exemple, demander un acompte). Un seuil bas, entre 0,20 et 0,30, est donc justifié. AUC sur le test : 0,67.

In [ ]:
g#schéma du kNN : x = délai, y = annulée ou honorée
import numpy as np

nouveau_delai = 1   # délai d'une nouvelle réservation (à changer pour tester)
k = 51

#les 51 réservations les plus proches en délai
distance = (modele['delai_reservation_j'] - nouveau_delai).abs()
voisines = distance.sort_values(kind='stable').index[:k]

#un petit décalage vertical aléatoire pour que les points ne se superposent pas
decalage = np.random.default_rng(42).uniform(-0.15, 0.15, len(modele))
y = modele['annule'] + decalage
est_voisine = modele.index.isin(voisines)

plt.figure(figsize=(9, 4))
plt.scatter(modele['delai_reservation_j'][~est_voisine], y[~est_voisine],
            s=8, color='lightgrey', label='autres réservations')
plt.scatter(modele['delai_reservation_j'][est_voisine], y[est_voisine],
            s=14, color='orange', label=f'les {k} voisines')
plt.axvline(nouveau_delai, color='black', linestyle='--', label='nouvelle réservation')
plt.yticks([0, 1], ['Honorée', 'Annulée'])
plt.xlabel('Délai de réservation (jours)')
plt.title('Le kNN regarde les réservations au délai le plus proche')
plt.legend()
plt.show()

nb_annulees = modele.loc[voisines, 'annule'].sum()
print(f'{nb_annulees} voisines annulées sur {k} → probabilité ≈ {nb_annulees / k:.0%}')

In [ ]:
# Calcul des probabilités d'annulation pour l'ensemble du jeu de données
modele['proba_annulation'] = knn.predict_proba(modele[['delai_reservation_j']])[:, 1]

# Découpage en 4 groupes de probabilités équitables (quartiles)
modele['groupe_proba'] = pd.qcut(modele['proba_annulation'], 4, duplicates='drop')

# Affichage de la table de validation pour comparer probabilité moyenne et taux réel d'annulation
display(modele.groupby('groupe_proba', observed=True).agg(
    proba_moyenne=('proba_annulation', 'mean'),
    taux_reel=('annule', 'mean'),
    reservations=('annule', 'size')).round(2))

In [ ]:
# Export pour le bloc C
export = modele[['reservation_id', 'hotel_id', 'categorie', 'segment_declare',
                 'delai_reservation_j', 'annule', 'proba_annulation']]
export.to_csv('q5_probabilites_annulation.csv', index=False)
print('Lignes exportées :', len(export))

### Conclusion Q5

- **On peut prédire l'annulation mieux que le hasard, mais de façon imparfaite** : AUC de 0,62 ± 0,06 en validation croisée, et 0,67 sur le test (0,5 = hasard, 1 = parfait).
- **Le seuil change tout** : à 0,50 (le seuil par défaut), le modèle ne trouve que 3 annulations sur 62. Avec un seuil de 0,30, il en trouve 33 (rappel 0,53), et quand il signale une annulation, il a raison 38 % du temps (contre 26 % au hasard).
- **Le délai porte toute l'information** : ajouter le nombre de nuits, de personnes ou le prix fait baisser l'AUC, et une simple règle à la main (délai > 45 jours) fait aussi bien que le kNN (F1 0,46 contre 0,44).
- **Pas de fuite d'information** : avec ou sans `nuitees_cumulees_client`, le résultat ne change pas. On l'exclut par prudence.
- **Les probabilités sont justes en moyenne** : par groupe de réservations, la probabilité prédite est proche du taux réel d'annulation.

### Recommandations Q5

- **Ne pas utiliser le modèle pour juger un client seul** : il se trompe trop souvent sur un cas individuel.
- **L'utiliser pour estimer le nombre d'annulations attendues**, par catégorie d'hôtel : c'est la base de la règle de surréservation (Q6).
- **Choisir un seuil bas (0,20 à 0,30)** si on veut repérer les réservations à risque, car rater une annulation (chambre vide) coûte plus cher qu'une fausse alerte.
- **En pratique, une règle simple suffit** : les réservations faites plus de 45 à 60 jours avant l'arrivée sont celles à surveiller (acompte, conditions d'annulation).
- **Pour améliorer la prédiction**, il faudrait d'autres données : historique d'annulation du client, type de tarif (remboursable ou non).